# 04 多年植被变化与气候分析

本实验使用2001—2025年MODIS EVI，从绿度、物候和气候关系三个层面描述区域变化。

研究问题：整体是在变绿还是变差？变化多快？生长季是否改变？与温度、降水是否存在统计关系？

## 分析框架

```text
多年MODIS EVI → 年均/峰值/振幅/积分 → 区域趋势与空间趋势
                         ↓                    ↓
                  同范围DL物候          显著变绿比例与速率
                         └──── 年度综合表 ────┘
                                   ↓
             TerraClimate气候 → 双线性插值对齐MODIS格网
                         ↓ 同一MODIS格点按共同年份配对
                    关系强度与显著性空间图
```


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import linregress, theilslopes, pearsonr, spearmanr
import pymannkendall as mk
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/vegetation_lab/modis_recovery.py").is_file())
sys.path.insert(0, str(ROOT / "src"))
from vegetation_lab import modis_recovery as lab
from vegetation_lab import modis_recovery_viz as viz

In [ ]:
START_YEAR, END_YEAR = 2001, 2025
INDEX = "evi"
REGION_ID = "beijing_download_bbox"
ALPHA = .05
CLIMATE_DIR = ROOT / "data/beijing_terraclimate"
DL_SPATIAL_DIR = None  # 可指定前一实验的完整DL运行目录；None时检查已有完整产品
RUN_PETTITT = False
RUN_STL = False
RUN_DIR = lab.new_run(ROOT)
settings = {"region_id": REGION_ID, "index": INDEX, "start_year": START_YEAR, "end_year": END_YEAR,
            "alpha": ALPHA, "QA": lab.QA, "climate_dir": str(CLIMATE_DIR),
            "DL_spatial_dir": str(DL_SPATIAL_DIR) if DL_SPATIAL_DIR else None,
            "RUN_PETTITT": RUN_PETTITT, "RUN_STL": RUN_STL}
(RUN_DIR / "configuration.json").write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding="utf8")
print("本次输出：", RUN_DIR)
print("气候输入：", CLIMATE_DIR)

In [ ]:
catalog, modis = lab.open_region(ROOT, REGION_ID, START_YEAR, END_YEAR, INDEX)
display(modis[[INDEX+"_analysis", "summaryqa_raw", "vi_quality"]])
display(catalog[["year", "manifest_exists", "tif_bytes"]])
print("首年time×lat×lon：", dict(modis.sizes))
print("后续年份按catalog逐年读取，未将首年冒充多年立方体。")

In [ ]:
coverage_preview = pd.DataFrame([{
    "起始年份": int(catalog.year.min()), "结束年份": int(catalog.year.max()), "年份数": len(catalog),
    "首年合成期数": modis.sizes["time"], "行": modis.sizes["lat"], "列": modis.sizes["lon"],
    "首年QA后有效比例": float(modis[INDEX+"_analysis"].notnull().mean()),
    "经度范围": f"{float(modis.lon.min()):.3f}—{float(modis.lon.max()):.3f}",
    "纬度范围": f"{float(modis.lat.min()):.3f}—{float(modis.lat.max()):.3f}"}])
display(coverage_preview)

## 1. 年度绿度指标

逐像元定义年均、最大、最小和振幅，再对像元指标作区域平均。
这里“区域平均年峰值”是各像元年最大值的平均，不是区域平均曲线的最大值。

$$A_y=VI_{max,y}-VI_{min,y},\qquad I_y\approx\sum_j\frac{VI_j+VI_{j+1}}{2}(t_{j+1}-t_j).$$

年均是有效合成期的等权平均，不称为全年逐日加权均值。

In [ ]:
# 先在同一固定位置看公式；空间循环仅复用这一年度指标定义。
example = modis[INDEX+"_analysis"].sel(lon=116.30, lat=40.60, method="nearest")
present = np.isfinite(example.values)
sample_vi = example.values[present]
sample_t = pd.DatetimeIndex(example.time.values).dayofyear.to_numpy(float)[present]
annual_mean = np.mean(sample_vi) if len(sample_vi) else np.nan
annual_max = np.max(sample_vi) if len(sample_vi) else np.nan
annual_min = np.min(sample_vi) if len(sample_vi) else np.nan
annual_amplitude = annual_max-annual_min
annual_integral = np.trapezoid(sample_vi, sample_t) if len(sample_vi) > 1 else np.nan
display(pd.DataFrame([dict(annual_mean=annual_mean, annual_max=annual_max, annual_min=annual_min,
                          annual_amplitude=annual_amplitude, annual_integral=annual_integral)]))

annual_cube, annual, regional_series, coverage = lab.prepare_annual(
    catalog, modis, INDEX, REGION_ID, RUN_DIR)
display(coverage)
display(annual)
display(annual_cube)

In [ ]:
viz.annual_lines(annual, ["annual_mean", "annual_max", "annual_integral"],
                 "完整下载ROI：年度绿度指标", RUN_DIR / "figures/01_greenness.png",
                 [INDEX.upper(), INDEX.upper(), INDEX.upper()+"·天"])
viz.annual_lines(annual, ["integral_span_days"], "积分覆盖天数：先检查观测支撑是否改变",
                 RUN_DIR / "figures/02_integral_support.png", ["天"])
print("学生问题：哪个指标变化最明显？积分变化是否同时伴随覆盖天数变化？")

## 2. 长期趋势：线性回归、Sen与Mann–Kendall

- 线性回归：$y=a+bt$，b为每年变化率，p检验斜率为0。
- Sen：所有年份对斜率的中位数，并报告95%置信区间；传入真实年份，不以行号代替年份。
- 原始MK：检验有无单调趋势，输出趋势、p和Tau；不是变化速率。

In [ ]:
valid = annual[["year", "annual_mean"]].dropna()
years = valid.year.to_numpy(float)
annual_mean_series = valid.annual_mean.to_numpy(float)
res = linregress(years, annual_mean_series)
slope, p_value = res.slope, res.pvalue
display(pd.DataFrame([{"Linear slope": slope, "p": p_value, "unit": INDEX.upper()+"/year"}]))

In [ ]:
sen = theilslopes(annual_mean_series, years, alpha=.95)
display(pd.DataFrame([{"Sen slope": sen.slope, "lower CI": sen.low_slope,
                      "upper CI": sen.high_slope, "unit": INDEX.upper()+"/year"}]))

In [ ]:
mk_result = mk.original_test(annual_mean_series, alpha=ALPHA)
display(pd.DataFrame([{"MK trend": mk_result.trend, "MK p": mk_result.p, "Tau": mk_result.Tau}]))

In [ ]:
trend_table = pd.DataFrame([{"metric": name,
                             "slope_unit": INDEX.upper()+"·day/year" if name == "annual_integral" else INDEX.upper()+"/year",
                             **lab.trend_summary(annual.year, annual[name], ALPHA)}
                            for name in lab.METRICS])
display(trend_table)
trend_table.to_csv(RUN_DIR / "greenness_trends.csv", index=False, encoding="utf-8-sig")

## 3. 多年物候：变绿是否伴随生长季改变？

In [ ]:
phenology = lab.regional_phenology(regional_series, REGION_ID, INDEX, RUN_DIR)
display(phenology)
viz.annual_lines(phenology, ["SOS", "EOS", "LOS"], "同一ROI平均曲线的DL物候",
                 RUN_DIR / "figures/03_phenology.png", ["DOY", "DOY", "天"])

In [ ]:
phenology_trends = []
for parameter in ["SOS", "EOS", "LOS"]:
    data = phenology[["year", parameter]].dropna()
    fit = linregress(data.year, data[parameter])
    result = lab.trend_summary(data.year, data[parameter], ALPHA)
    phenology_trends.append({"parameter": parameter, "slope_day_per_year": fit.slope,
                             "Sen_day_per_year": result["sen_slope"], "MK_p": result["mk_p"],
                             "MK_trend": result["mk_trend"]})
display(pd.DataFrame(phenology_trends))
pd.DataFrame(phenology_trends).to_csv(RUN_DIR / "phenology_trends.csv", index=False, encoding="utf-8-sig")

In [ ]:
annual_table = annual.merge(phenology, on="year", validate="one_to_one", how="left")
annual_table["region_id"] = REGION_ID
display(annual_table)
annual_table.to_csv(RUN_DIR / "annual_integrated.csv", index=False, encoding="utf-8-sig")

## 4. 空间恢复监测

令像元$i$的多年VI线性斜率为$R_i$，单位VI/年。
本课“恢复率”仅指**显著绿度增加像元比例**：

$$100\times N(slope>0, p<0.05)/N(\text{具有可计算slope和p的像元}).$$

分子分母使用同一指标和同一格网集合。平均恢复速率指显著增加集合的平均斜率，另报全体可计算像元平均斜率。
无有效像元时比例为空；无显著增加像元时比例为0、该集合平均斜率为空。

In [ ]:
greenness_maps = {name: lab.pixelwise_trend(annual_cube.year.values, annual_cube[name])
                  for name in ["annual_mean", "annual_max"]}
for name, product in greenness_maps.items():
    product.attrs.update(index=INDEX, region_id=REGION_ID, test="OLS two-sided t; unadjusted p")
    product.to_netcdf(RUN_DIR / f"{name}_trend.nc", engine="h5netcdf")
display(greenness_maps["annual_mean"])

In [ ]:
dl_cube, spatial_status = lab.load_spatial_dl(ROOT, annual_cube.year.values.tolist(), INDEX,
                                             annual_cube, DL_SPATIAL_DIR)
print(spatial_status["state"])
phenology_maps = None
if dl_cube is None:
    print(spatial_status["message"])
else:
    phenology_maps = {name: lab.pixelwise_trend(dl_cube.year.values, dl_cube.sel(metric=name))
                      for name in ["SOS", "EOS", "LOS"]}
    for name, product in phenology_maps.items():
        product.to_netcdf(RUN_DIR / f"{name}_trend.nc", engine="h5netcdf")
(RUN_DIR / "spatial_phenology_status.json").write_text(json.dumps(spatial_status, ensure_ascii=False, indent=2), encoding="utf8")

In [ ]:
recovery_rows = []
for metric, product in greenness_maps.items():
    slope = product.slope.values
    p = product.p.values
    valid = np.isfinite(slope) & np.isfinite(p)
    significant_increase = valid & (slope > 0) & (p < ALPHA)
    recovery_rate = 100*significant_increase.sum()/valid.sum() if valid.any() else np.nan
    mean_recovery_speed = slope[significant_increase].mean() if significant_increase.any() else np.nan
    mean_all_slope = slope[valid].mean() if valid.any() else np.nan
    recovery_rows.append({"metric": metric, "valid_pixels": int(valid.sum()),
                          "significant_increase_pixels": int(significant_increase.sum()),
                          "significant_increase_percent": recovery_rate,
                          "mean_slope_significant_increase": mean_recovery_speed,
                          "mean_slope_all_computable": mean_all_slope})
recovery = pd.DataFrame(recovery_rows)
recovery["slope_unit"] = INDEX.upper()+"/year"
display(recovery)
recovery.to_csv(RUN_DIR / "recovery_summary.csv", index=False, encoding="utf-8-sig")

In [ ]:
viz.recovery_maps(greenness_maps, phenology_maps, ALPHA, INDEX, RUN_DIR / "figures/04_recovery_maps.png")

## 5. 气候—物候空间异质性：相关不等于因果

直接读取2001—2024年TerraClimate月尺度GeoTIFF，在原生1/24°格网上派生春季温度、秋季温度和年降水。
温度按月天数加权；降水按12个月累计。

In [ ]:
climate = lab.load_terraclimate_spatial(CLIMATE_DIR, START_YEAR, END_YEAR)
climate.to_netcdf(RUN_DIR / "terraclimate_annual_native_grid.nc", engine="h5netcdf")
climate_on_modis = lab.interpolate_climate_to_modis_grid(climate, annual_cube, method="bilinear")
climate_on_modis.to_netcdf(RUN_DIR / "terraclimate_annual_bilinear_on_modis_grid.nc", engine="h5netcdf")
print("气候立方体：", dict(climate.sizes), climate.attrs["spatial_resolution"])
print("双线性插值后气候格网：", dict(climate_on_modis.sizes))

In [ ]:
spatial_climate_results = {
    "spring_temp vs SOS": None,
    "autumn_temp vs EOS": None,
    "annual_precip vs annual_mean EVI": lab.spatial_correlation(climate_on_modis.annual_precip, annual_cube.annual_mean),
}
if dl_cube is not None:
    spatial_climate_results["spring_temp vs SOS"] = lab.spatial_correlation(
        climate_on_modis.spring_temp, dl_cube.sel(metric="SOS"))
    spatial_climate_results["autumn_temp vs EOS"] = lab.spatial_correlation(
        climate_on_modis.autumn_temp, dl_cube.sel(metric="EOS"))
for name, result in spatial_climate_results.items():
    if result is not None:
        result.to_netcdf(RUN_DIR / (name.replace(" ", "_") + ".nc"), engine="h5netcdf")

In [ ]:
spatial_climate_summary = lab.spatial_correlation_summary(spatial_climate_results, ALPHA)
display(spatial_climate_summary)
spatial_climate_summary.to_csv(RUN_DIR / "spatial_climate_correlation_summary.csv", index=False, encoding="utf-8-sig")
viz.climate_correlation_maps(spatial_climate_results, ALPHA, RUN_DIR / "figures/05_spatial_climate_correlation.png")
(RUN_DIR / "climate_status.json").write_text(json.dumps({
    "source": str(CLIMATE_DIR), "years": climate.year.values.tolist(),
    "source_grid": "native TerraClimate 1/24 degree", "target_grid": "MODIS 349x466",
    "interpolation": "bilinear; alignment only, no new climate spatial information",
    "spatial_phenology_500m": "loaded" if dl_cube is not None else "pending_full_grid_DL",
}, ensure_ascii=False, indent=2), encoding="utf8")

## 可选拓展：Pettitt与STL（默认不执行）

Pettitt寻找潜在单次水平改变，报告切分前最后一个观测年份与近似p；不是工程发生年识别器。
实现用秩累计统计量及常用近似p，结果受并列值、自相关与边界影响，不把小p解释成因果。
STL先将区域合成期曲线显式按时间插值到日，再取月均，period=12；它是重建后的月序列分解。

In [ ]:
if RUN_PETTITT:
    change = lab.pettitt(annual.year, annual.annual_mean)
    display(pd.DataFrame([change]))
    (RUN_DIR / "pettitt.json").write_text(json.dumps(change, indent=2), encoding="utf8")
else:
    print("Pettitt拓展未执行")
if RUN_STL:
    decomposition = lab.monthly_stl(regional_series)
    display(decomposition.head())
    decomposition.to_csv(RUN_DIR / "stl_monthly.csv", encoding="utf-8-sig")
    viz.stl_plot(decomposition, RUN_DIR / "figures/06_stl.png")
else:
    print("STL拓展未执行")

## 6. 综合讨论与作业

1. 2001—2025年样区总体是否变绿？年均、峰值、振幅和积分支持同一判断吗？
2. 线性斜率与Sen斜率是否一致？MK说明的是速率还是趋势证据？
3. 显著增加像元占多少？分母、单位、多重检验和地表组成如何影响解释？
4. SOS、EOS和LOS是否变化？区域平均曲线日期能否当作平均像元日期？
5. 绿度增加是否伴随生长季延长？积分覆盖变化是否影响结论？
6. 2001—2024年气候相关系数的正负说明什么？2025年缺气候对样本数有何影响？
7. 能否仅凭这些结果断言生态工程导致恢复？没有处理区和对照区时还缺什么？
8. 若做更可靠的因果分析，还需要哪些工程时间/边界、土地覆盖、管理和气候资料？